In [ ]:
!pip install transformers sentencepiece torch


In [ ]:
import pandas as pd
import re, random
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import T5ForConditionalGeneration, AutoTokenizer


device = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)
device


In [ ]:
def parse_dialog(raw):
    raw = raw.strip()
    if raw.startswith("[") and raw.endswith("]"):
        raw = raw[1:-1]

    parts = re.split(r"'([^']*)'", raw)

    utts = []
    for p in parts:
        p = p.strip()
        if p and not p.startswith("[") and not p.startswith("]"):
            utts.append(p)
    return utts


def parse_int_list(raw):
    raw = raw.strip()
    if raw.startswith("[") and raw.endswith("]"):
        raw = raw[1:-1]
    raw = raw.strip()
    if not raw:
        return []
    return list(map(int, raw.split()))


In [ ]:
df = pd.read_csv("train.csv")

df["dialog"]  = df["dialog"].apply(parse_dialog)
df["act"]     = df["act"].apply(parse_int_list)
df["emotion"] = df["emotion"].apply(parse_int_list)

df.head()


In [ ]:
pairs = []

for _, row in df.iterrows():
    dialog = row["dialog"]
    acts   = row["act"]
    emos   = row["emotion"]

    L = min(len(dialog), len(acts), len(emos))

    for i in range(L - 1):
        src = dialog[i]
        tgt = dialog[i+1]
        act = acts[i+1]
        emo = emos[i+1]
        pairs.append((src, tgt, act, emo))

len(pairs)


In [ ]:
random.shuffle(pairs)
pairs = pairs[:6000]

train_pairs = pairs[:5500]
test_pairs  = pairs[5500:]
len(train_pairs), len(test_pairs)


In [ ]:
tokenizer = AutoTokenizer.from_pretrained("t5-small")


In [ ]:
class PairDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        src, tgt, act, emo = self.pairs[idx]
        enc = tokenizer(src, padding="max_length", truncation=True, max_length=64, return_tensors="pt")
        dec = tokenizer(tgt, padding="max_length", truncation=True, max_length=64, return_tensors="pt")
        return (
            enc.input_ids.squeeze(),
            enc.attention_mask.squeeze(),
            dec.input_ids.squeeze(),
            torch.tensor(act),
            torch.tensor(emo)
        )

train_ds = PairDataset(train_pairs)
test_ds  = PairDataset(test_pairs)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)


In [ ]:
seq2seq = T5ForConditionalGeneration.from_pretrained("t5-small")
seq2seq.to(device)


In [ ]:
optimizer = torch.optim.AdamW(seq2seq.parameters(), lr=2e-4)

for epoch in range(8):
    seq2seq.train()
    for ids, mask, tgt, act, emo in train_loader:
        ids = ids.to(device)
        mask = mask.to(device)
        tgt = tgt.to(device)

        out = seq2seq(input_ids=ids, attention_mask=mask, labels=tgt)
        loss = out.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

    print("epoch", epoch, "loss", loss.item())


In [ ]:
def gen_seq2seq(text):
    inp = tokenizer(text, return_tensors="pt").input_ids.to(device)
    out = seq2seq.generate(inp, max_length=40)
    print("> ", tokenizer.decode(out[0], skip_special_tokens=True))




In [ ]:
gen_seq2seq("How are you?")
gen_seq2seq("What are your hobbies?")
gen_seq2seq("Where do you live?")
gen_seq2seq("Do you like sports?")


In [ ]:
latent_dim = 64
from transformers.modeling_outputs import BaseModelOutput

class CVAE(nn.Module):
    def __init__(self):
        super().__init__()

        self.t5 = T5ForConditionalGeneration.from_pretrained("t5-small")

        self.encoder = self.t5.encoder
        self.decoder = self.t5.decoder

        self.lm_head = nn.Linear(512, self.t5.config.vocab_size)

        self.mu = nn.Linear(512, latent_dim)
        self.logvar = nn.Linear(512, latent_dim)

        self.z_proj = nn.Linear(latent_dim, 512)

    def sample(self, mu, logvar):
        eps = torch.randn_like(mu)
        return mu + torch.exp(0.5 * logvar) * eps

    def forward(self, ids, mask, tgt):
        enc = self.encoder(
            input_ids=ids,
            attention_mask=mask,
        )

        h = enc.last_hidden_state[:, 0, :]      # (batch, 512)

        mu     = self.mu(h)                     # (batch, 64)
        logvar = self.logvar(h)                 # (batch, 64)
        z      = self.sample(mu, logvar)        # (batch, 64)

        z = self.z_proj(z)                      # (batch, 512)

        
        z = z.unsqueeze(1)                      # (batch, 1, 512)
        z = z.repeat(1, tgt.size(1), 1)         # (batch, seq_len, 512)

      
        enc_states = BaseModelOutput(last_hidden_state=z)

        outputs = self.decoder(
            input_ids=tgt,
            encoder_hidden_states=enc_states.last_hidden_state,
        )

        logits = self.lm_head(outputs.last_hidden_state)

        recon_loss = nn.CrossEntropyLoss(ignore_index=0)(
            logits.view(-1, logits.size(-1)),
            tgt.view(-1),
        )

        kld = -0.5 * torch.mean(
            1 + logvar - mu.pow(2) - logvar.exp()
        )

        return recon_loss + 0.1 * kld

    @torch.no_grad()
    def generate(self, input_text, tokenizer, device="cpu", n=3, max_length=40):
        inp = tokenizer(input_text, return_tensors="pt").to(device)

        enc = self.encoder(
            input_ids=inp.input_ids,
            attention_mask=inp.attention_mask
        )
        h = enc.last_hidden_state[:, 0, :]

        mu     = self.mu(h)
        logvar = self.logvar(h)

        outputs = []

        for _ in range(n):
            # sample latent
            z = self.sample(mu, logvar)
            z = self.z_proj(z)
            z = z.unsqueeze(1)                  # (batch, 1, 512)
            enc_states = BaseModelOutput(last_hidden_state=z)

            
            out = self.t5.generate(
                encoder_outputs=enc_states,
                max_length=max_length
            )

            outputs.append(
                tokenizer.decode(out[0], skip_special_tokens=True)
            )

        return outputs

In [ ]:
cvae = CVAE().to(device)


In [ ]:
optim = torch.optim.AdamW(cvae.parameters(), lr=2e-4)

for epoch in range(8):
    for ids, mask, tgt, act, emo in train_loader:
        ids = ids.to(device)
        mask = mask.to(device)
        tgt = tgt.to(device)

        loss = cvae(ids, mask, tgt)
        loss.backward()
        optim.step()
        optim.zero_grad()

    print("epoch", epoch, "loss", loss.item())


In [ ]:
def gen_cvae(text, n=3):
    outputs = cvae.generate(text, tokenizer, device=device, n=n)
    for o in outputs:
        print("> ", o)

gen_cvae("How are you?")


In [ ]:
gen_cvae("How are you?")
gen_cvae("What are your hobbies?")
gen_cvae("Where do you live?")
gen_cvae("Do you like sports?")